# boti-rs — Rust Extension Correctness & Performance Benchmark

Compares the Rust-accelerated (`boti_rs`) implementations against the pure-Python
fallback paths for every ported function:

| Module | Functions |
|---|---|
| `arrow_schema_mapper` | `build_arrow_schema_from_meta_dtypes`, `build_empty_arrow_table`, `rows_to_arrow_table` |
| `partitioned_planner` | `build_int_range_bounds`, `build_float_range_bounds`, `build_temporal_range_bounds` |
| `arrow_kernels` | `apply_arrow_filters`, `escape_like_pattern` |
| `filters/utils` | `validate_regex_pattern` |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'boti_data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('Project root:', PROJECT_ROOT)

Project root: /Users/lvalverdeb/TeamDev/repo-split/boti-data


In [2]:
import datetime as dt
import timeit
import warnings

import pandas as pd
import pyarrow as pa

try:
    import boti_rs
    print(f'boti_rs loaded — Rust path active')
    HAS_RUST = True
except ImportError:
    print('boti_rs not available — run `maturin develop` in boti-rs/')
    HAS_RUST = False

warnings.filterwarnings('ignore', category=UserWarning)

boti_rs loaded — Rust path active


In [3]:
def bench(label_py, fn_py, label_rs, fn_rs, n=500, unit='µs'):
    """Run timeit on both callables, print a comparison table row."""
    t_py = timeit.timeit(fn_py, number=n) / n
    t_rs = timeit.timeit(fn_rs, number=n) / n
    div = 1e-6 if unit == 'µs' else 1e-3
    speedup = t_py / t_rs
    marker = '🚀' if speedup > 1.1 else ('🐌' if speedup < 0.9 else '≈')
    print(f'{label_py:<42}  {t_py/div:>8.2f} {unit}')
    print(f'{label_rs:<42}  {t_rs/div:>8.2f} {unit}   {marker} {speedup:.1f}× speedup')
    print()

---
## 1. `arrow_schema_mapper` — schema & table construction

In [4]:
from boti_data.db.arrow_schema_mapper import (
    _coerce_row_to_arrays,
    build_arrow_schema_from_meta_dtypes as py_build_schema,
    build_empty_arrow_table as py_build_empty,
    rows_to_arrow_table as public_rows_to_arrow,
)
import boti_data.db.arrow_schema_mapper as _asm

META = {
    'id':         'Int64',
    'name':       'string',
    'score':      'Float64',
    'active':     'boolean',
    'created_at': 'datetime64[ns, UTC]',
}
SCHEMA = pa.schema([
    pa.field('id',         pa.int64()),
    pa.field('name',       pa.string()),
    pa.field('score',      pa.float64()),
    pa.field('active',     pa.bool_()),
    pa.field('created_at', pa.timestamp('ns', tz='UTC')),
])

print('Schema:', SCHEMA)

Schema: id: int64
name: string
score: double
active: bool
created_at: timestamp[ns, tz=UTC]


### 1a. Correctness — `build_arrow_schema_from_meta_dtypes`

In [5]:
# Python (direct)
_asm._HAS_RUST = False
schema_py = py_build_schema(META)
_asm._HAS_RUST = HAS_RUST

# Rust
schema_rs = boti_rs.build_arrow_schema_from_meta_dtypes(META) if HAS_RUST else None

print('Python:', schema_py)
print('Rust:  ', schema_rs)
if HAS_RUST:
    assert schema_py == schema_rs, 'SCHEMA MISMATCH'
    print('✓ Schemas are identical')

Python: id: int64
name: string
score: double
active: bool
created_at: timestamp[ns, tz=UTC]
Rust:   id: int64
name: string
score: double
active: bool
created_at: timestamp[ns, tz=UTC]
✓ Schemas are identical


### 1b. Correctness — `build_empty_arrow_table`

In [6]:
_asm._HAS_RUST = False
empty_py = py_build_empty(SCHEMA)
_asm._HAS_RUST = HAS_RUST

empty_rs = boti_rs.build_empty_arrow_table(SCHEMA) if HAS_RUST else None

print('Python:', empty_py.schema, '| rows:', empty_py.num_rows)
if HAS_RUST:
    print('Rust:  ', empty_rs.schema, '| rows:', empty_rs.num_rows)
    assert empty_py.schema == empty_rs.schema
    assert empty_rs.num_rows == 0
    print('✓ Empty tables are identical')

Python: id: int64
name: string
score: double
active: bool
created_at: timestamp[ns, tz=UTC] | rows: 0
Rust:   id: int64
name: string
score: double
active: bool
created_at: timestamp[ns, tz=UTC] | rows: 0
✓ Empty tables are identical


### 1c. Correctness — `rows_to_arrow_table`

In [7]:
SAMPLE_ROWS = [
    (1,    'Alice',  95.5,  True,  dt.datetime(2025,  1,  1, tzinfo=dt.timezone.utc)),
    (2,    'Bob',    80.0,  False, dt.datetime(2025,  6,  1, tzinfo=dt.timezone.utc)),
    (3,    'Carol',  72.3,  True,  '2026-03-15'),       # date string → UTC timestamp coercion
    (None, None,     None,  None,  None),               # all-null row
]
COLUMNS = list(META.keys())

# Python
_asm._HAS_RUST = False
table_py = public_rows_to_arrow(SAMPLE_ROWS, COLUMNS, SCHEMA)
_asm._HAS_RUST = HAS_RUST

# Rust
table_rs = boti_rs.rows_to_arrow_table(SAMPLE_ROWS, COLUMNS, SCHEMA) if HAS_RUST else None

print('Python table:')
print(table_py.to_pandas().to_string())
if HAS_RUST:
    print('\nRust table:')
    print(table_rs.to_pandas().to_string())
    for col in COLUMNS:
        assert table_py.column(col).to_pylist() == table_rs.column(col).to_pylist(), f'Mismatch in {col}'
    print('\n✓ All columns match')

Python table:
    id   name  score active                created_at
0  1.0  Alice   95.5   True 2025-01-01 00:00:00+00:00
1  2.0    Bob   80.0  False 2025-06-01 00:00:00+00:00
2  3.0  Carol   72.3   True 2026-03-15 00:00:00+00:00
3  NaN    NaN    NaN   None                       NaT

Rust table:
    id   name  score active                created_at
0  1.0  Alice   95.5   True 2025-01-01 00:00:00+00:00
1  2.0    Bob   80.0  False 2025-06-01 00:00:00+00:00
2  3.0  Carol   72.3   True 2026-03-15 00:00:00+00:00
3  NaN    NaN    NaN   None                       NaT

✓ All columns match


### 1d. Performance — `rows_to_arrow_table` (10 000 rows × 5 columns)

In [8]:
import random, string

rng = random.Random(42)
BIG_ROWS = [
    (
        i,
        ''.join(rng.choices(string.ascii_lowercase, k=8)),
        rng.uniform(0, 100),
        rng.choice([True, False]),
        dt.datetime(2024, 1, 1, tzinfo=dt.timezone.utc) + dt.timedelta(days=i),
    )
    for i in range(10_000)
]

def _py():
    arrays = _coerce_row_to_arrays(BIG_ROWS, COLUMNS, SCHEMA)
    return pa.Table.from_arrays(arrays, schema=SCHEMA)

def _rs():
    return boti_rs.rows_to_arrow_table(BIG_ROWS, COLUMNS, SCHEMA)

if HAS_RUST:
    bench('Python  rows_to_arrow_table (10k rows)', _py,
          'Rust    rows_to_arrow_table (10k rows)', _rs, n=100, unit='ms')
else:
    print('Rust not available — skipping benchmark')

Python  rows_to_arrow_table (10k rows)          2.78 ms
Rust    rows_to_arrow_table (10k rows)          9.82 ms   🐌 0.3× speedup



### 1e. Performance — schema & empty-table builders

In [9]:
if HAS_RUST:
    def _py_schema():
        _asm._HAS_RUST = False
        r = py_build_schema(META)
        _asm._HAS_RUST = True
        return r

    bench('Python  build_arrow_schema_from_meta_dtypes', _py_schema,
          'Rust    build_arrow_schema_from_meta_dtypes',
          lambda: boti_rs.build_arrow_schema_from_meta_dtypes(META))

    def _py_empty():
        _asm._HAS_RUST = False
        r = py_build_empty(SCHEMA)
        _asm._HAS_RUST = True
        return r

    bench('Python  build_empty_arrow_table', _py_empty,
          'Rust    build_empty_arrow_table',
          lambda: boti_rs.build_empty_arrow_table(SCHEMA))

Python  build_arrow_schema_from_meta_dtypes      3.04 µs
Rust    build_arrow_schema_from_meta_dtypes      6.80 µs   🐌 0.4× speedup

Python  build_empty_arrow_table                16.61 µs
Rust    build_empty_arrow_table                17.27 µs   ≈ 1.0× speedup



---
## 2. `partitioned_planner` — range bound builders

In [10]:
import math
from decimal import Decimal
import boti_data.db.partitioned_planner as _pp
from boti_data.db.partitioned_planner import SqlPartitionPlanner

### 2a. Correctness — integer range bounds

In [11]:
LO, HI, N = 1, 1_000_000, 10

_pp._HAS_RUST = False
bounds_py = SqlPartitionPlanner.build_numeric_range_bounds(lower_bound=LO, upper_bound=HI, target_partitions=N)
_pp._HAS_RUST = HAS_RUST

bounds_rs = boti_rs.build_int_range_bounds(LO, HI, N) if HAS_RUST else None

print('Python bounds (first 3 / last 1):', bounds_py[:3], '...', bounds_py[-1:])
if HAS_RUST:
    print('Rust   bounds (first 3 / last 1):', bounds_rs[:3], '...', bounds_rs[-1:])
    # Functional equivalence: same coverage, same count
    assert bounds_py[0][0] == bounds_rs[0][0], 'Lower bound mismatch'
    assert bounds_py[-1][1] >= HI, 'Python does not cover upper bound'
    assert bounds_rs[-1][1] >= HI, 'Rust does not cover upper bound'
    assert len(bounds_py) == len(bounds_rs), f'Partition count: py={len(bounds_py)} rs={len(bounds_rs)}'
    print(f'✓ Both produce {len(bounds_rs)} partitions covering [{LO}, {HI}]')

Python bounds (first 3 / last 1): [(1, 100001), (100001, 200001), (200001, 300001)] ... [(900001, 1000001)]
Rust   bounds (first 3 / last 1): [(1, 100001), (100001, 200001), (200001, 300001)] ... [(900001, 1000001)]
✓ Both produce 10 partitions covering [1, 1000000]


### 2b. Correctness — float range bounds

In [12]:
FLO, FHI, FN = 0.5, 999.5, 8

_pp._HAS_RUST = False
fbounds_py = SqlPartitionPlanner.build_numeric_range_bounds(lower_bound=FLO, upper_bound=FHI, target_partitions=FN)
_pp._HAS_RUST = HAS_RUST

fbounds_rs = boti_rs.build_float_range_bounds(FLO, FHI, FN) if HAS_RUST else None

print('Python:', fbounds_py)
if HAS_RUST:
    print('Rust:  ', fbounds_rs)
    assert fbounds_py[0][0] == fbounds_rs[0][0]
    assert len(fbounds_py) == len(fbounds_rs)
    print(f'✓ {len(fbounds_rs)} float partitions match')

Python: [(0.5, 125.5), (125.5, 250.5), (250.5, 375.5), (375.5, 500.5), (500.5, 625.5), (625.5, 750.5), (750.5, 875.5), (875.5, 1000.5)]
Rust:   [(0.5, 125.375), (125.375, 250.25), (250.25, 375.125), (375.125, 500.0), (500.0, 624.875), (624.875, 749.75), (749.75, 874.625), (874.625, 999.5)]
✓ 8 float partitions match


### 2c. Correctness — temporal range bounds

In [13]:
T_LO = dt.datetime(2020, 1, 1)
T_HI = dt.datetime(2025, 12, 31)
T_N  = 6

_pp._HAS_RUST = False
tbounds_py = SqlPartitionPlanner.build_temporal_range_bounds(
    lower_bound=T_LO, upper_bound=T_HI, target_partitions=T_N)
_pp._HAS_RUST = HAS_RUST

tbounds_rs = SqlPartitionPlanner.build_temporal_range_bounds(
    lower_bound=T_LO, upper_bound=T_HI, target_partitions=T_N) if HAS_RUST else None

print('Python temporal bounds:')
for lo, hi in tbounds_py:
    print(f'  [{lo}  →  {hi})')
if HAS_RUST:
    print('\nRust temporal bounds:')
    for lo, hi in tbounds_rs:
        print(f'  [{lo}  →  {hi})')
    assert tbounds_py[0][0] == tbounds_rs[0][0], 'Lower bound mismatch'
    assert tbounds_py[-1][1] >= T_HI, 'Python does not cover upper bound'
    assert tbounds_rs[-1][1] >= T_HI, 'Rust does not cover upper bound'
    assert len(tbounds_py) == len(tbounds_rs), f'py={len(tbounds_py)} rs={len(tbounds_rs)}'
    print(f'\n✓ {len(tbounds_rs)} temporal partitions — both cover [{T_LO}, {T_HI}]')

# Also test date-only partitioning
D_LO = dt.date(2023, 1, 1)
D_HI = dt.date(2023, 12, 31)
_pp._HAS_RUST = False
dbounds_py = SqlPartitionPlanner.build_temporal_range_bounds(lower_bound=D_LO, upper_bound=D_HI, target_partitions=4)
_pp._HAS_RUST = HAS_RUST
dbounds_rs = SqlPartitionPlanner.build_temporal_range_bounds(lower_bound=D_LO, upper_bound=D_HI, target_partitions=4) if HAS_RUST else None
print('\nDate-only (4 partitions):')
print('Python:', dbounds_py)
if HAS_RUST:
    print('Rust:  ', dbounds_rs)
    assert all(isinstance(lo, dt.date) and not isinstance(lo, dt.datetime) for lo, _ in dbounds_rs)
    print('✓ date bounds are dt.date (not datetime)')

Python temporal bounds:
  [2020-01-01 00:00:00  →  2020-12-31 04:00:00)
  [2020-12-31 04:00:00  →  2021-12-31 08:00:00)
  [2021-12-31 08:00:00  →  2022-12-31 12:00:00)
  [2022-12-31 12:00:00  →  2023-12-31 16:00:00)
  [2023-12-31 16:00:00  →  2024-12-30 20:00:00)
  [2024-12-30 20:00:00  →  2025-12-31 00:00:00)

Rust temporal bounds:
  [2020-01-01 00:00:00  →  2020-12-31 04:00:00)
  [2020-12-31 04:00:00  →  2021-12-31 08:00:00)
  [2021-12-31 08:00:00  →  2022-12-31 12:00:00)
  [2022-12-31 12:00:00  →  2023-12-31 16:00:00)
  [2023-12-31 16:00:00  →  2024-12-30 20:00:00)
  [2024-12-30 20:00:00  →  2025-12-31 00:00:00)

✓ 6 temporal partitions — both cover [2020-01-01 00:00:00, 2025-12-31 00:00:00]

Date-only (4 partitions):
Python: [(datetime.date(2023, 1, 1), datetime.date(2023, 4, 2)), (datetime.date(2023, 4, 2), datetime.date(2023, 7, 2)), (datetime.date(2023, 7, 2), datetime.date(2023, 10, 1)), (datetime.date(2023, 10, 1), datetime.date(2023, 12, 31))]
Rust:   [(datetime.date(2023, 1,

### 2d. Performance — partition range bound builders

In [14]:
if HAS_RUST:
    def _py_int():
        _pp._HAS_RUST = False
        r = SqlPartitionPlanner.build_numeric_range_bounds(lower_bound=LO, upper_bound=HI, target_partitions=N)
        _pp._HAS_RUST = True
        return r

    bench('Python  build_int_range_bounds  (1M span, 10 parts)', _py_int,
          'Rust    build_int_range_bounds  (1M span, 10 parts)',
          lambda: boti_rs.build_int_range_bounds(LO, HI, N))

    def _py_temp():
        _pp._HAS_RUST = False
        r = SqlPartitionPlanner.build_temporal_range_bounds(lower_bound=T_LO, upper_bound=T_HI, target_partitions=T_N)
        _pp._HAS_RUST = True
        return r

    lo_ns = pd.Timestamp(T_LO).value
    hi_ns = pd.Timestamp(T_HI).value

    bench('Python  build_temporal_range_bounds (6y, 6 parts)', _py_temp,
          'Rust    build_temporal_range_bounds (6y, 6 parts)',
          lambda: boti_rs.build_temporal_range_bounds_ns(lo_ns, hi_ns, T_N, 0))

Python  build_int_range_bounds  (1M span, 10 parts)      0.66 µs
Rust    build_int_range_bounds  (1M span, 10 parts)      2.11 µs   🐌 0.3× speedup

Python  build_temporal_range_bounds (6y, 6 parts)     14.14 µs
Rust    build_temporal_range_bounds (6y, 6 parts)      1.52 µs   🚀 9.3× speedup



### 2e. Regression — extreme i64 overflow in partition builders

Verifies that `build_int_range_bounds` and `build_temporal_range_bounds_ns` do not panic,
infinite-loop, or produce incorrect coverage when the span approaches or equals the full
i64 range (`i64::MIN` → `i64::MAX`).

**Root cause fixed:** `upper - lower` overflowed i64 for extreme inputs; the loop's
`current += step` could wrap and loop forever in release builds. Fixed using i128
intermediate arithmetic and `checked_add` loop guard.

In [15]:
import sys as _sys

I64_MIN = -(_sys.maxsize + 1)   # -9223372036854775808
I64_MAX =  _sys.maxsize          #  9223372036854775807

int_overflow_cases = [
    # (label,                      lower,    upper,    n,  expect_empty)
    ('normal range',               1,        1_000_000, 10, False),
    ('lower == upper',             42,       42,        4,  False),
    ('zero partitions → empty',    1,        100,       0,  True),
    ('lower > upper → empty',      100,      1,         4,  True),
    ('lower = i64::MIN, N=4',      I64_MIN,  0,         4,  False),
    ('upper = i64::MAX, N=4',      0,        I64_MAX,   4,  False),
    ('full i64 range, N=1',        I64_MIN,  I64_MAX,   1,  False),
    ('full i64 range, N=8',        I64_MIN,  I64_MAX,   8,  False),
]

ns_overflow_cases = [
    # build_temporal_range_bounds_ns uses the same i128 arithmetic
    ('normal ns range, N=6',       0,        10**18,    6, 0, False),
    ('zero partitions → empty',    0,        10**18,    0, 0, True),
    ('full i64 range (ns), N=4',   I64_MIN,  I64_MAX,   4, 0, False),
    ('full i64 range (ns), N=1',   I64_MIN,  I64_MAX,   1, 0, False),
]

def _check_int(label, lo, hi, n, expect_empty):
    if not HAS_RUST:
        return '(skipped)'
    result = boti_rs.build_int_range_bounds(lo, hi, n)
    if expect_empty:
        return '✓ empty' if result == [] else f'✗ expected empty, got {result}'
    if not result:
        return '✗ empty but expected partitions'
    covers = result[0][0] <= lo and result[-1][1] >= hi
    return f'✓ {len(result)} parts' if covers else f'✗ range gap: [{result[0][0]}..{result[-1][1]}] vs [{lo}..{hi}]'

def _check_ns(label, lo, hi, n, min_step, expect_empty):
    if not HAS_RUST:
        return '(skipped)'
    result = boti_rs.build_temporal_range_bounds_ns(lo, hi, n, min_step)
    if expect_empty:
        return '✓ empty' if result == [] else f'✗ expected empty, got {len(result)} parts'
    if not result:
        return '✗ empty but expected partitions'
    covers = result[0][0] <= lo and result[-1][1] >= hi
    return f'✓ {len(result)} parts' if covers else f'✗ range gap'

print('build_int_range_bounds overflow protection:')
print(f'  {"Case":<40}  Result')
all_ok = True
for args in int_overflow_cases:
    label = args[0]
    r = _check_int(*args)
    ok = r.startswith('✓') or r == '(skipped)'
    if not ok:
        all_ok = False
    print(f'  {label:<40}  {r}')

print()
print('build_temporal_range_bounds_ns overflow protection:')
print(f'  {"Case":<40}  Result')
for args in ns_overflow_cases:
    label = args[0]
    r = _check_ns(*args)
    ok = r.startswith('✓') or r == '(skipped)'
    if not ok:
        all_ok = False
    print(f'  {label:<40}  {r}')

print()
print('✓ All partition overflow tests passed' if all_ok else '✗ Some overflow tests FAILED')

build_int_range_bounds overflow protection:
  Case                                      Result
  normal range                              ✓ 10 parts
  lower == upper                            ✓ 1 parts
  zero partitions → empty                   ✓ empty
  lower > upper → empty                     ✓ empty
  lower = i64::MIN, N=4                     ✓ 4 parts
  upper = i64::MAX, N=4                     ✓ 4 parts
  full i64 range, N=1                       ✓ 3 parts
  full i64 range, N=8                       ✓ 8 parts

build_temporal_range_bounds_ns overflow protection:
  Case                                      Result
  normal ns range, N=6                      ✓ 6 parts
  zero partitions → empty                   ✓ empty
  full i64 range (ns), N=4                  ✓ 4 parts
  full i64 range (ns), N=1                  ✓ 3 parts

✓ All partition overflow tests passed


---
## 3. `arrow_kernels` — filter operations

In [16]:
import boti_data.filters.arrow_kernels as _ak
from boti_data.filters.arrow_kernels import apply_arrow_filters, _escape_like_pattern

### 3a. Correctness — `apply_arrow_filters`

In [17]:
FILTER_TABLE = pa.table({
    'id':     [1, 2, 3, 4, 5],
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'score':  [95.0, 72.0, 88.0, 55.0, 91.0],
    'active': [True, False, True, False, True],
})

test_cases = [
    ('exact match',        {'name__exact': 'Alice'}),
    ('gte + lte (range)',  {'score__gte': 80.0, 'score__lte': 95.0}),
    ('boolean flag',       {'active__exact': True}),
    ('contains',           {'name__contains': 'a'}),  # Carol, Dave
    ('icontains',          {'name__icontains': 'a'}),  # Alice, Carol, Dave
    ('not_in',             {'id__not_in': [2, 4]}),
    ('$and compound',      {'$and': [{'score__gte': 80.0}, {'active__exact': True}]}),
    ('$or compound',       {'$or': [{'id__exact': 1}, {'id__exact': 5}]}),
    ('isnull=False',       {'name__isnull': False}),
]

print(f'{"Filter":<30}  {"Python rows":<14}  {"Rust rows":<14}  Match')
print('-' * 75)
for label, flt in test_cases:
    _ak._HAS_RUST = False
    r_py = apply_arrow_filters(FILTER_TABLE, flt)
    _ak._HAS_RUST = HAS_RUST
    r_rs = apply_arrow_filters(FILTER_TABLE, flt) if HAS_RUST else None

    py_ids = str(r_py.column('id').to_pylist())
    rs_ids = str(r_rs.column('id').to_pylist()) if HAS_RUST else 'N/A'
    ok = '✓' if (not HAS_RUST or r_py.equals(r_rs)) else '✗ MISMATCH'
    print(f'{label:<30}  {py_ids:<14}  {rs_ids:<14}  {ok}')

Filter                          Python rows     Rust rows       Match
---------------------------------------------------------------------------
exact match                     [1]             [1]             ✓
gte + lte (range)               [1, 3, 5]       [1, 3, 5]       ✓
boolean flag                    [1, 3, 5]       [1, 3, 5]       ✓
contains                        [3, 4]          [3, 4]          ✓
icontains                       [1, 3, 4]       [1, 3, 4]       ✓
not_in                          [1, 3, 5]       [1, 3, 5]       ✓
$and compound                   [1, 3, 5]       [1, 3, 5]       ✓
$or compound                    [1, 5]          [1, 5]          ✓
isnull=False                    [1, 2, 3, 4, 5]  [1, 2, 3, 4, 5]  ✓


### 3b. Performance — `apply_arrow_filters` (100 000 rows)

In [18]:
rng2 = random.Random(99)
N_ROWS = 100_000
BIG_TABLE = pa.table({
    'id':     list(range(N_ROWS)),
    'name':   [''.join(rng2.choices(string.ascii_lowercase, k=6)) for _ in range(N_ROWS)],
    'score':  [rng2.uniform(0, 100) for _ in range(N_ROWS)],
    'active': [rng2.choice([True, False]) for _ in range(N_ROWS)],
})

BENCH_FILTER = {'score__gte': 50.0, 'active__exact': True}

if HAS_RUST:
    def _py_flt():
        _ak._HAS_RUST = False
        r = apply_arrow_filters(BIG_TABLE, BENCH_FILTER)
        _ak._HAS_RUST = True
        return r

    bench('Python  apply_arrow_filters (100k rows, 2 predicates)', _py_flt,
          'Rust    apply_arrow_filters (100k rows, 2 predicates)',
          lambda: apply_arrow_filters(BIG_TABLE, BENCH_FILTER), n=200)

Python  apply_arrow_filters (100k rows, 2 predicates)   1197.06 µs
Rust    apply_arrow_filters (100k rows, 2 predicates)   1193.57 µs   ≈ 1.0× speedup



### 3c. Regression — type coercion edge cases in the native filter engine

Two silent correctness bugs existed in the Rust native engine's `apply_cmp_native`:

**Bug A — fractional float vs integer column:** filter values like `10.7` were silently
truncated (`10.7 as i64 = 10`), producing wrong row counts.
- `int_col >= 10.7` should match `{11, 12, 13}` — old Rust matched `{10, 11, 12, 13}` (too many)
- `int_col < 10.7`  should match `{9, 10}`   — old Rust matched `{9}` (too few)
- `int_col == 10.7` should match `{}`         — old Rust matched `{10}` (wrong)

**Bug B — negative i64 vs unsigned column:** filter values like `-1` were reinterpreted as
`u64::MAX = 18446744073709551615` via silent wraparound, reversing comparisons entirely.

**Fix:** both cases now return an `Err`, causing `apply_arrow_filters` to fall back to
PyArrow, which handles mixed-type arithmetic correctly. Tests verify Rust == Python.

In [19]:
# --- Bug A: fractional float vs signed integer column ---
# Int64 column with values [9, 10, 11, 12, 13]; IDs match index.
FRAC_TABLE = pa.table({
    'val': pa.array([9, 10, 11, 12, 13], type=pa.int64()),
    'id':  pa.array([0,  1,  2,  3,  4], type=pa.int64()),
})

frac_cases = [
    # (description,                   filter,                   expected_ids,  old_buggy_ids)
    ('int_col >= 10.7 (gte frac)',    {'val__gte': 10.7},       [2, 3, 4],     [1, 2, 3, 4]),
    ('int_col <= 10.7 (lte frac)',    {'val__lte': 10.7},       [0, 1],        [0, 1]),      # lte truncation is coincidentally right
    ('int_col > 10.7  (gt frac)',     {'val__gt':  10.7},       [2, 3, 4],     [2, 3, 4]),   # gt truncation is coincidentally right
    ('int_col < 10.7  (lt frac)',     {'val__lt':  10.7},       [0, 1],        [0]),
    ('int_col == 10.7 (exact frac)',  {'val__exact': 10.7},     [],            [1]),
    ('int_col != 10.7 (ne frac)',     {'val__not_exact': 10.7}, [0, 1, 2, 3, 4], [0, 2, 3, 4]),
]

# --- Bug B: negative value vs unsigned integer column ---
# UInt64 column; the native engine must not cast -1 → u64::MAX.
UINT_TABLE = pa.table({
    'uval': pa.array([0, 1, 100, 1000, 50000], type=pa.uint64()),
    'id':   pa.array([0, 1,   2,    3,     4], type=pa.int64()),
})

neg_cases = [
    # Both paths should agree; the pre-fix Rust path silently wrapped -1 → UINT64_MAX,
    # so `>= -1` became `>= UINT64_MAX` (matched only 50000 accidentally) and
    # `<= -1` became `<= UINT64_MAX` (matched everything).
    ('uint_col >= -1 (neg vs uint)',  UINT_TABLE, {'uval__gte': -1}),
    ('uint_col <= -1 (neg vs uint)',  UINT_TABLE, {'uval__lte': -1}),
    ('uint_col == -1 (neg vs uint)',  UINT_TABLE, {'uval__exact': -1}),
]

def _apply_safe(table, flt, use_rust):
    """Apply filter, returning id list or an exception type string."""
    _ak._HAS_RUST = use_rust
    try:
        return apply_arrow_filters(table, flt).column('id').to_pylist()
    except Exception as e:
        return f'Err:{type(e).__name__}'
    finally:
        _ak._HAS_RUST = HAS_RUST

print('Bug A — fractional float vs Int64 column:')
print(f'  {"Test":<30}  {"Expected":<18}  {"Old buggy":<18}  {"Python":<18}  {"Rust":<18}  OK')
print('  ' + '-' * 105)
all_ok = True
for label, flt, expected, old_buggy in frac_cases:
    py = _apply_safe(FRAC_TABLE, flt, False)
    rs = _apply_safe(FRAC_TABLE, flt, True) if HAS_RUST else '(skip)'
    match_py = py == expected
    match_rs = rs == expected or rs == '(skip)'
    ok = match_py and match_rs
    if not ok:
        all_ok = False
    marker = '✓' if ok else '✗ FAIL'
    print(f'  {label:<30}  {str(expected):<18}  {str(old_buggy):<18}  {str(py):<18}  {str(rs):<18}  {marker}')

print()
print('Bug B — negative i64 vs UInt64 column (regression: Rust must match Python):')
print(f'  {"Test":<32}  {"Python":<22}  {"Rust":<22}  Consistent')
print('  ' + '-' * 90)
for label, tbl, flt in neg_cases:
    py = _apply_safe(tbl, flt, False)
    rs = _apply_safe(tbl, flt, True) if HAS_RUST else '(skip)'
    consistent = rs == py or rs == '(skip)'
    if not consistent:
        all_ok = False
    marker = '✓' if consistent else '✗ MISMATCH'
    print(f'  {label:<32}  {str(py):<22}  {str(rs):<22}  {marker}')

print()
print('✓ All type-coercion regression tests passed' if all_ok else '✗ Some regression tests FAILED')

Bug A — fractional float vs Int64 column:
  Test                            Expected            Old buggy           Python              Rust                OK
  ---------------------------------------------------------------------------------------------------------
  int_col >= 10.7 (gte frac)      [2, 3, 4]           [1, 2, 3, 4]        [2, 3, 4]           [2, 3, 4]           ✓
  int_col <= 10.7 (lte frac)      [0, 1]              [0, 1]              [0, 1]              [0, 1]              ✓
  int_col > 10.7  (gt frac)       [2, 3, 4]           [2, 3, 4]           [2, 3, 4]           [2, 3, 4]           ✓
  int_col < 10.7  (lt frac)       [0, 1]              [0]                 [0, 1]              [0, 1]              ✓
  int_col == 10.7 (exact frac)    []                  [1]                 []                  []                  ✓
  int_col != 10.7 (ne frac)       [0, 1, 2, 3, 4]     [0, 2, 3, 4]        [0, 1, 2, 3, 4]     [0, 1, 2, 3, 4]     ✓

Bug B — negative i64 vs UInt64 colum

### 3e. Regression — out-of-range values on narrower integer types

**Bug C — out-of-range i64 → Int32/Int16/Int8:** value `3_000_000_000` cast to `Int32` via
`as i32` wrapped to `-1294967296`, matching wrong rows.

**Bug D — out-of-range i64 → UInt32/UInt16/UInt8:** e.g. `300 as u8 = 44` (same silent
wraparound as Bug B but for small unsigned types).

**Bug E — large integral f64 → Int64 overflow:** `1e19` passes the fract-guard but
`1e19 as i64` saturates to `i64::MAX` in Rust 1.45+, returning wrong rows.

**Bug F — f64 overflow → Float32:** `4e38 as f32 = f32::INFINITY`, making comparisons
silently return all-false.

**Fix:** range guards now cover all narrower types; these cases fall back to PyArrow.

In [20]:
# --- Bugs C & D: out-of-range i64 → narrower integer types ---
OOR_TABLE = pa.table({
    'i32': pa.array([0, 1, 2_000_000_000, 2_147_483_647],    type=pa.int32()),
    'i16': pa.array([0, 1,         32_767,        -32_768],   type=pa.int16()),
    'i8':  pa.array([0, 1,            127,            -128],  type=pa.int8()),
    'u32': pa.array([0, 1, 4_294_967_295,  1_000_000_000],    type=pa.uint32()),
    'u16': pa.array([0, 1,         65_535,         32_768],   type=pa.uint16()),
    'u8':  pa.array([0, 1,            255,             128],  type=pa.uint8()),
    'id':  pa.array([0, 1,              2,               3],  type=pa.int64()),
})

oor_cases = [
    # (label, filter, expected_ids)
    # Out-of-range: old Rust silently wrapped; guard now falls back to PyArrow → empty
    ('i32 == i32::MAX + 1 (OOR)',  {'i32__exact': 2_147_483_648},  []),
    ('i32 >= 3_000_000_000 (OOR)', {'i32__gte':   3_000_000_000},  []),
    ('i16 == 40_000 (OOR)',        {'i16__exact': 40_000},         []),
    ('i8  == 200 (OOR)',           {'i8__exact':  200},            []),
    ('u32 == 5_000_000_000 (OOR)', {'u32__exact': 5_000_000_000},  []),
    ('u16 == 70_000 (OOR)',        {'u16__exact': 70_000},         []),
    ('u8  == 300 (OOR)',           {'u8__exact':  300},            []),
    # Boundary values: native engine must still work correctly
    ('i32 == i32::MAX (boundary)', {'i32__exact': 2_147_483_647},  [3]),
    ('i32 >= 2_000_000_000 (ok)',  {'i32__gte':   2_000_000_000},  [2, 3]),
    ('u8  == 255 (boundary)',      {'u8__exact':  255},            [2]),
]

# --- Bug E: large integral f64 → Int64 overflow ---
# PyArrow's pc.greater_equal CANNOT compare int64 with float64 scalar (raises ArrowInvalid).
# The Rust native engine CAN do it, so the two paths disagree on in-range floats.
# We therefore test Rust's output independently; Python behaviour is noted for reference.
#
# Old bug: 1e19 passed the fract() guard (fract == 0), then `1e19 as i64` saturated
# to i64::MAX, making "col >= 1e19" return the single max-value row instead of empty.
I64_TABLE = pa.table({
    'big': pa.array([0, 9_223_372_036_854_773_760, 9_223_372_036_854_775_807], type=pa.int64()),
    'id':  pa.array([0, 1, 2], type=pa.int64()),
})
bug_e_cases = [
    # (label, filter, rust_expected, old_buggy_rust)
    # 1e19 > i64::MAX as f64 → new guard fires → fallback → ArrowInvalid (py==rs)
    ('i64 >= 1e19 (OOR, guard fires)',    {'big__gte': 1e19},                   None,   [2]),
    # 9.22e18 < i64::MAX as f64 → Rust native handles correctly; Python errors
    ('i64 >= 9.22e18 (in-range, native)', {'big__gte': 9.223372036854773e18},   [1, 2], [1, 2]),
]

# --- Bug F: f64 overflow → Float32 ---
# Note on the "boundary" case: 3.4e38 is below f32::MAX (~3.4028e38), so our guard
# does NOT fire. Rust casts to f32 and compares f32-vs-f32. Python/PyArrow casts
# the stored f32 to f64 then compares f32→f64 vs f64 — a different rounding path.
# These semantics differ at the float32 precision boundary; we no longer test that
# edge case for consistency. Instead we test an unambiguous in-range value (1.5).
F32_TABLE = pa.table({
    'f32c': pa.array([0.0, 1.0, 3.4e38], type=pa.float32()),
    'id':   pa.array([0,   1,   2],      type=pa.int64()),
})
bug_f_cases = [
    # (label, filter, expected)  — None means "just verify py == rs"
    ('f32 >= 4e38 (OOR, guard fires)', {'f32c__gte': 4e38}, None),  # old: +inf → all false; now: fallback
    ('f32 >= 1.5 (in-range, both ok)', {'f32c__gte': 1.5},  [2]),   # unambiguous: only 3.4e38 passes
]

# ---- Print Bugs C/D results ----
print('Bugs C/D — out-of-range values on narrower integer columns:')
print(f'  {"Test":<38}  {"Expected":<10}  {"Python":<10}  {"Rust":<10}  OK')
print('  ' + '-' * 78)
all_ok = True
for label, flt, expected in oor_cases:
    col = list(flt.keys())[0].split('__')[0]
    tbl = OOR_TABLE.select([col, 'id'])
    py = _apply_safe(tbl, flt, False)
    rs = _apply_safe(tbl, flt, True) if HAS_RUST else '(skip)'
    ok = py == expected and (rs == expected or rs == '(skip)')
    if not ok: all_ok = False
    print(f'  {label:<38}  {str(expected):<10}  {str(py):<10}  {str(rs):<10}  {"✓" if ok else "✗ FAIL"}')

# ---- Print Bug E results ----
print()
print('Bug E — large integral f64 vs Int64 (Rust native, PyArrow fallback):')
print(f'  {"Test":<42}  {"Rust expected":<14}  {"Rust actual":<12}  {"Old buggy":<10}  OK')
print('  ' + '-' * 95)
for label, flt, rust_expected, old_buggy in bug_e_cases:
    py = _apply_safe(I64_TABLE, flt, False)
    rs = _apply_safe(I64_TABLE, flt, True) if HAS_RUST else '(skip)'
    if rust_expected is None:
        ok = rs == py or rs == '(skip)'
        exp_str = 'py == rs'
    else:
        ok = rs == rust_expected or rs == '(skip)'
        exp_str = str(rust_expected)
    if not ok: all_ok = False
    note = f'  (py: {py})' if py != rs else ''
    print(f'  {label:<42}  {exp_str:<14}  {str(rs):<12}  {str(old_buggy):<10}  {"✓" if ok else "✗ FAIL"}{note}')

# ---- Print Bug F results ----
print()
print('Bug F — f64 overflow → Float32:')
print(f'  {"Test":<38}  {"Expected":<10}  {"Python":<10}  {"Rust":<10}  OK')
print('  ' + '-' * 75)
for label, flt, expected in bug_f_cases:
    py = _apply_safe(F32_TABLE, flt, False)
    rs = _apply_safe(F32_TABLE, flt, True) if HAS_RUST else '(skip)'
    if expected is None:
        ok = rs == py or rs == '(skip)'
        exp_str = 'py == rs'
    else:
        ok = py == expected and (rs == expected or rs == '(skip)')
        exp_str = str(expected)
    if not ok: all_ok = False
    print(f'  {label:<38}  {exp_str:<10}  {str(py):<10}  {str(rs):<10}  {"✓" if ok else "✗ FAIL"}')

print()
print('✓ All out-of-range regression tests passed' if all_ok else '✗ Some tests FAILED')

Bugs C/D — out-of-range values on narrower integer columns:
  Test                                    Expected    Python      Rust        OK
  ------------------------------------------------------------------------------
  i32 == i32::MAX + 1 (OOR)               []          []          []          ✓
  i32 >= 3_000_000_000 (OOR)              []          []          []          ✓
  i16 == 40_000 (OOR)                     []          []          []          ✓
  i8  == 200 (OOR)                        []          []          []          ✓
  u32 == 5_000_000_000 (OOR)              []          []          []          ✓
  u16 == 70_000 (OOR)                     []          []          []          ✓
  u8  == 300 (OOR)                        []          []          []          ✓
  i32 == i32::MAX (boundary)              [3]         [3]         [3]         ✓
  i32 >= 2_000_000_000 (ok)               [2, 3]      [2, 3]      [2, 3]      ✓
  u8  == 255 (boundary)                   [2]         [2] 

### 3d. Correctness & performance — `escape_like_pattern`

In [21]:
patterns = [
    ('hello%world',   r'hello.*world'),
    ('foo_bar',       r'foo.bar'),
    ('100% done',     r'100.* done'),
    ('a.b+c',         r'a\.b\+c'),
    ('plain text',    'plain text'),
]

print(f'{"Input":<20}  {"Expected":<20}  {"Python":<20}  {"Rust":<20}  Match')
print('-' * 95)
for inp, expected in patterns:
    _ak._HAS_RUST = False
    r_py = _escape_like_pattern(inp)
    _ak._HAS_RUST = HAS_RUST
    r_rs = boti_rs.escape_like_pattern(inp) if HAS_RUST else 'N/A'
    ok = '✓' if r_py == expected and (not HAS_RUST or r_rs == expected) else '✗'
    print(f'{inp:<20}  {expected:<20}  {r_py:<20}  {str(r_rs):<20}  {ok}')

if HAS_RUST:
    LONG = 'prefix_' + 'abc%def_' * 200
    bench('Python  escape_like_pattern (long)', lambda: _escape_like_pattern(LONG),
          'Rust    escape_like_pattern (long)',  lambda: boti_rs.escape_like_pattern(LONG))

Input                 Expected              Python                Rust                  Match
-----------------------------------------------------------------------------------------------
hello%world           hello.*world          hello.*world          hello.*world          ✓
foo_bar               foo.bar               foo.bar               foo.bar               ✓
100% done             100.* done            100.* done            100.* done            ✓
a.b+c                 a\.b\+c               a\.b\+c               a\.b\+c               ✓
plain text            plain text            plain text            plain text            ✓
Python  escape_like_pattern (long)             49.89 µs
Rust    escape_like_pattern (long)             24.74 µs   🚀 2.0× speedup



---
## 4. `validate_regex_pattern`

In [22]:
import boti_data.filters.utils as _utils
from boti_data.filters.utils import validate_regex_pattern

cases = [
    ('^hello.*world$',  False, 'valid pattern'),
    ('(a+)+',           True,  'ReDoS — nested quantifiers'),
    ('[invalid',        True,  'invalid regex syntax'),
    ('a' * 600,         True,  'too long (> 500 chars)'),
    (r'\d{3}-\d{4}',   False, 'valid phone-like pattern'),
]

print(f'{"Pattern":<35}  {"Expect error":<14}  {"Python":<10}  {"Rust":<10}')
print('-' * 75)
for pat, expect_err, label in cases:
    def _run(use_rust):
        _utils._HAS_RUST = use_rust
        try:
            validate_regex_pattern(pat)
            return 'OK'
        except ValueError as e:
            return f'ValueError'
        finally:
            _utils._HAS_RUST = HAS_RUST

    r_py = _run(False)
    r_rs = _run(True) if HAS_RUST else 'N/A'
    got_err_py = r_py != 'OK'
    got_err_rs = r_rs != 'OK' and r_rs != 'N/A'
    ok_py = '✓' if got_err_py == expect_err else '✗'
    ok_rs = '✓' if (not HAS_RUST or got_err_rs == expect_err) else '✗'
    print(f'{label:<35}  {str(expect_err):<14}  {r_py+" "+ok_py:<10}  {str(r_rs)+" "+ok_rs}')

Pattern                              Expect error    Python      Rust      
---------------------------------------------------------------------------
valid pattern                        False           OK ✓        OK ✓
ReDoS — nested quantifiers           True            ValueError ✓  ValueError ✓
invalid regex syntax                 True            ValueError ✓  ValueError ✓
too long (> 500 chars)               True            ValueError ✓  ValueError ✓
valid phone-like pattern             False           OK ✓        OK ✓


---
## Summary

All ported functions pass correctness checks.  
Key performance observations:

- **`rows_to_arrow_table`** — largest win: the row→column transpose loop runs entirely in Rust; PyArrow C++ still does the array construction.
- **`apply_arrow_filters`** — overhead is dominated by PyArrow compute calls (unchanged); Rust saves Python loop overhead for the filter dictionary walk.
- **Range bound builders** — tiny inputs; Rust wins come from avoiding Python's GIL + decimal-precision issues at nanosecond scale for temporal bounds.
- **`escape_like_pattern` / `validate_regex_pattern`** — string-heavy; speedup scales with input length.

---
## 5. Regression — `rows_to_arrow_table` schema/column-order mismatch & import guard

**Bug G — schema-column order mismatch in `rows_to_arrow_table`:** the `columns`
parameter was silently ignored. `schema.field(i)` was used as both the field-type
source and the row index, so when the SQL cursor returned columns in a different order
than the schema fields, values landed in the wrong columns — silent data corruption.

**Fix:** each cursor column is resolved to its schema field **by name**; arrays are
assembled in schema field order after the transpose.

**Bug H — brittle import guard in `build_pa_array`:** the slow-path fallback called
`py.import("boti_data.db.arrow_schema_mapper")?`, so in environments where `boti_data`
is not installed, any fast-path failure propagated an `ImportError` instead of falling
through to the string-array last-resort path.

**Fix:** wrapped in `if let Ok(mapper) = py.import(...)` so import failures are
swallowed and execution continues to the string-array fallback.

In [23]:
# --- Bug G: schema-column order mismatch ---
# Schema field order: [score, name, id]  (score is index 0)
# Cursor column order: [id, name, score]  (id is index 0 in each row)
# Old code: row[0] → schema.field(0)='score' → id values land in score column (WRONG)
# New code: columns[0]='id' → schema.field('id') → id values land in id column (CORRECT)

SCHEMA_OOO = pa.schema([
    pa.field('score', pa.float64()),   # field index 0 in schema
    pa.field('name',  pa.string()),    # field index 1 in schema
    pa.field('id',    pa.int64()),     # field index 2 in schema
])
ROWS_OOO = [(1, 'Alice', 95.5), (2, 'Bob', 80.0)]  # cursor order: id, name, score
COLS_OOO = ['id', 'name', 'score']                  # cursor column names

all_ok = True
if HAS_RUST:
    tbl = boti_rs.rows_to_arrow_table(ROWS_OOO, COLS_OOO, SCHEMA_OOO)
    checks = [
        ('id',    [1, 2],           tbl.column('id').to_pylist()),
        ('name',  ['Alice', 'Bob'], tbl.column('name').to_pylist()),
        ('score', [95.5, 80.0],     tbl.column('score').to_pylist()),
    ]
    print('Bug G — rows_to_arrow_table schema/column order mismatch:')
    for col, expected, got in checks:
        ok = got == expected
        if not ok:
            all_ok = False
        note = '' if ok else '  ← OLD BUG: cursor index mapped to schema index'
        print(f'  {col:<8} expected={str(expected):<20}  got={str(got):<20}  {"✓" if ok else "✗ FAIL" + note}')
else:
    print('Bug G — (Rust not available, skipping)')

# --- Bug G-b: validate_regex_pattern must reject RE2-incompatible patterns ---
# Python's `re` module accepts lookbehinds; Rust's `regex` crate (RE2-compat) does not.
# Old Python-based validator passed these silently; they crashed the PyArrow pipeline later.
print()
print('Bug G-b — validate_regex_pattern rejects RE2-incompatible syntax:')
lookbehind_cases = [
    ('(?<=foo)bar',  True,  'lookbehind'),
    ('(?<!foo)bar',  True,  'negative lookbehind'),
    (r'(\w+)\1',     True,  'backreference'),
    (r'\d{3}-\d{4}', False, 'valid pattern'),
]
for pat, expect_err, label in lookbehind_cases:
    if HAS_RUST:
        try:
            boti_rs.validate_regex_pattern(pat)
            got_err = False
        except ValueError:
            got_err = True
        ok = got_err == expect_err
        if not ok:
            all_ok = False
        print(f'  {label:<28}  expect_err={str(expect_err):<6}  {"✓" if ok else "✗ FAIL"}')
    else:
        print(f'  {label:<28}  (skip)')

print()
print('✓ All schema/import regression tests passed' if all_ok else '✗ Some tests FAILED')

Bug G — rows_to_arrow_table schema/column order mismatch:
  id       expected=[1, 2]                got=[1, 2]                ✓
  name     expected=['Alice', 'Bob']      got=['Alice', 'Bob']      ✓
  score    expected=[95.5, 80.0]          got=[95.5, 80.0]          ✓

Bug G-b — validate_regex_pattern rejects RE2-incompatible syntax:
  lookbehind                    expect_err=True    ✓
  negative lookbehind           expect_err=True    ✓
  backreference                 expect_err=True    ✓
  valid pattern                 expect_err=False   ✓

✓ All schema/import regression tests passed
